In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
import os


# 1. DATA INGESTION & PREPROCESSING


def load_and_clean_data(sentiment_path, trader_path):
    print("Loading datasets...")
    df_sentiment = pd.read_csv(sentiment_path)
    df_trades = pd.read_csv(trader_path)

    # Standardize Dates
    df_sentiment['date'] = pd.to_datetime(df_sentiment['date']).dt.date

    # The historical data uses 'Timestamp IST' with mixed formats
    df_trades['Date'] = pd.to_datetime(df_trades['Timestamp IST'], format='mixed', dayfirst=True).dt.date

    # Clean financial columns
    df_trades['Closed PnL'] = pd.to_numeric(df_trades['Closed PnL'], errors='coerce').fillna(0)
    df_trades['Size USD'] = pd.to_numeric(df_trades['Size USD'], errors='coerce').fillna(0)

    return df_sentiment, df_trades


# 2. DATA INTEGRATION


def merge_datasets(df_sentiment, df_trades):
    print("Merging datasets on Date...")
    # Inner join to ensure every trade has a sentiment classification
    df_merged = pd.merge(df_trades, df_sentiment, left_on='Date', right_on='date', how='inner')
    df_merged['Is Win'] = df_merged['Closed PnL'] > 0
    return df_merged


# 3. STATISTICAL SIGNIFICANCE TESTING


def perform_statistical_tests(df):
    print("\n--- Statistical Significance Testing ---")

    # Isolate PnL arrays for Extreme Greed vs Extreme Fear
    greed_pnl = df[df['classification'] == 'Extreme Greed']['Closed PnL']
    fear_pnl = df[df['classification'] == 'Extreme Fear']['Closed PnL']

    # Perform Independent T-Test
    t_stat, p_value = stats.ttest_ind(greed_pnl, fear_pnl, equal_var=False)

    print(f"T-Statistic: {t_stat:.4f}")
    print(f"P-Value: {p_value:.4e}")

    if p_value < 0.05:
        print("Conclusion: The difference in average PnL between Extreme Greed and Extreme Fear is STATISTICALLY SIGNIFICANT.")
    else:
        print("Conclusion: There is no statistically significant difference in PnL between these market conditions.")


# 4. INTERACTIVE VISUALIZATIONS


def generate_interactive_dashboards(df):
    print("\nGenerating Interactive Dashboards...")
    sentiment_order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']

    # Chart 1: Average PnL by Sentiment
    sentiment_stats = df.groupby('classification').agg(avg_pnl=('Closed PnL', 'mean')).reset_index()
    fig1 = px.bar(sentiment_stats, x='classification', y='avg_pnl',
                  category_orders={'classification': sentiment_order},
                  title="Average PnL per Trade by Market Sentiment",
                  color='avg_pnl', color_continuous_scale='RdYlGn')
    fig1.write_html("/content/interactive_pnl_by_sentiment.html")

    # Chart 2: Buy vs Sell Asymmetry
    direction_stats = df.groupby(['classification', 'Side']).agg(
        avg_pnl=('Closed PnL', 'mean'), win_rate=('Is Win', 'mean'), trade_count=('Account', 'count')
    ).reset_index()
    fig2 = px.bar(direction_stats, x='classification', y='avg_pnl', color='Side', barmode='group',
                  category_orders={'classification': sentiment_order},
                  title="Contrarian Edge: Average PnL for BUY vs SELL across Sentiments",
                  hover_data=['win_rate', 'trade_count'])
    fig2.write_html("/content/interactive_contrarian_edge.html")
    print("Dashboards saved to /content/")


# 5. INSIGHTS


def generate_advanced_insights(df):
    print("\n" + "="*50)
    print("INSIGHTS")
    print("="*50)

    # --- INSIGHT 1: Average Bet Size ---
    print("\n1. Average Bet Size by Sentiment:")
    size_analysis = df.groupby('classification').agg(
        Average_Bet_Size=('Size USD', 'mean'), Total_Trades=('Account', 'count')
    ).reset_index().sort_values(by='Average_Bet_Size', ascending=False)
    print(size_analysis.to_string(index=False))
    size_analysis.to_csv("/content/insight_bet_size.csv", index=False)

    # --- INSIGHT 2: Average Win vs. Average Loss ---
    print("\n2. Average Win vs. Average Loss:")
    win_loss = df.groupby(['classification', 'Is Win']).agg(Average_Amount=('Closed PnL', 'mean')).reset_index()
    win_loss['Result'] = win_loss['Is Win'].map({True: 'WIN', False: 'LOSS'})
    win_loss = win_loss[['classification', 'Result', 'Average_Amount']]
    print(win_loss.to_string(index=False))
    win_loss.to_csv("/content/insight_win_loss.csv", index=False)

    # --- INSIGHT 3: Expected Value (EV) ---
    print("\n3. Expected Value (EV) per Trade (Buy vs Sell):")
    ev_df = df.groupby(['classification', 'Side']).apply(
        lambda x: pd.Series({
            'Win_Rate': (x['Closed PnL'] > 0).mean(),
            'Avg_Win': x[x['Closed PnL'] > 0]['Closed PnL'].mean() if not x[x['Closed PnL'] > 0].empty else 0,
            'Avg_Loss': x[x['Closed PnL'] <= 0]['Closed PnL'].mean() if not x[x['Closed PnL'] <= 0].empty else 0
        })
    ).reset_index()
    ev_df['Expected_Value'] = (ev_df['Win_Rate'] * ev_df['Avg_Win']) + ((1 - ev_df['Win_Rate']) * ev_df['Avg_Loss'])
    ev_df = ev_df.sort_values(by='Expected_Value', ascending=False)
    print(ev_df[['classification', 'Side', 'Win_Rate', 'Expected_Value']].to_string(index=False))
    ev_df.to_csv("/content/insight_expected_value.csv", index=False)

    # --- INSIGHT 4: Temporal Analysis (Day of Week) ---
    print("\n4. Temporal Analysis (Profitability by Day of Week):")
    df['Day_of_Week'] = pd.to_datetime(df['Date']).dt.day_name()
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_analysis = df.groupby('Day_of_Week').agg(
        Total_Trades=('Account', 'count'), Avg_PnL=('Closed PnL', 'mean'), Win_Rate=('Is Win', 'mean')
    ).reindex(day_order).reset_index()
    print(day_analysis.to_string(index=False))
    day_analysis.to_csv("/content/insight_day_of_week.csv", index=False)


# 6. EXECUTION


if __name__ == "__main__":
    # File paths for Colab environment
    SENTIMENT_FILE = "/content/fear_greed_index.csv"
    TRADES_FILE = "/content/historical_data.csv"

    try:
        # Step 1 & 2: Load and Merge
        sentiment_df, trades_df = load_and_clean_data(SENTIMENT_FILE, TRADES_FILE)
        merged_df = merge_datasets(sentiment_df, trades_df)

        # Step 3: Statistical Test
        perform_statistical_tests(merged_df)

        # Step 4: Dashboards
        generate_interactive_dashboards(merged_df)

        # Step 5: Insights (EV, Trade Size, Time, Win/Loss)
        generate_advanced_insights(merged_df)

        # Final Export of the main summary
        final_summary = merged_df.groupby(['classification', 'Side']).agg(
            total_trades=('Account', 'count'),
            avg_pnl=('Closed PnL', 'mean'),
            win_rate=('Is Win', 'mean')
        ).reset_index()
        final_summary.to_csv("/content/sentiment_trading_summary.csv", index=False)

        print("\n" + "="*50)
        print("ALL ANALYSIS COMPLETE!")
        print("Check your Colab file folder for your generated CSVs and HTML dashboards.")
        print("="*50)

    except FileNotFoundError as e:
        print(f"Error: {e}. Please ensure the CSV files are uploaded to your /content/ directory.")

Loading datasets...
Merging datasets on Date...

--- Statistical Significance Testing ---
T-Statistic: 3.8512
P-Value: 1.1778e-04
Conclusion: The difference in average PnL between Extreme Greed and Extreme Fear is STATISTICALLY SIGNIFICANT.

Generating Interactive Dashboards...
Dashboards saved to /content/

INSIGHTS

1. Average Bet Size by Sentiment:
classification  Average_Bet_Size  Total_Trades
          Fear       7816.109931         61837
         Greed       5736.884375         50303
  Extreme Fear       5349.731843         21400
       Neutral       4782.732661         37686
 Extreme Greed       3112.251565         39992

2. Average Win vs. Average Loss:
classification Result  Average_Amount
  Extreme Fear   LOSS      -47.243417
  Extreme Fear    WIN      173.424767
 Extreme Greed   LOSS      -12.660059
 Extreme Greed    WIN      160.593269
          Fear   LOSS      -16.572501
          Fear    WIN      151.840935
         Greed   LOSS      -34.211857
         Greed    WIN     

/tmp/ipykernel_2419/1432277155.py:116: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

